# Build the fsl training image - fast, one run

Reads project/region from `configs/pipeline_config.json` (no re-typing), then
builds and pushes the image in one cell. Default tag is **`candidate`** - the
tag the test pipeline uses. When a candidate passes the gate, the pipeline
retags it to `production`; the production pipeline uses `production`.

**Just run the two cells below.** Works whether this notebook sits in `scripts/`
or the project root - it finds the root automatically.

Requires: Docker on the machine (your Workbench has it), and a Dockerfile in
the project root (you already have it).

## 1. Setup - find root, read config, decide the tag

In [1]:
import json
import os
from pathlib import Path


def find_project_root(start=None):
    """Walk up from CWD to find the folder with Dockerfile + configs/."""
    p = Path(start or Path.cwd()).resolve()
    for _ in range(6):
        if (p / "Dockerfile").exists() and (p / "configs" / "pipeline_config.json").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(
        "Couldn't find project root (needs Dockerfile + configs/pipeline_config.json). "
        "Run this from inside the project."
    )


ROOT = find_project_root()
os.chdir(ROOT)                       # docker build needs to run from the root (build context)
print("Project root:", ROOT)

with open("configs/pipeline_config.json") as f:
    CFG = json.load(f)

PROJECT = CFG["project"]
REGION  = CFG["region"]

# --- the one knob you might change ---
VERSION = "candidate"   # the tag the test pipeline uses; promoted to "production" on gate pass
REPO    = "fsl-images"
IMAGE   = "train"

IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT}/{REPO}/{IMAGE}:{VERSION}"

# guard: unedited placeholder project would produce a broken URI
if "YOUR-PROJECT" in PROJECT or PROJECT.endswith("-ID"):
    raise ValueError(
        f"config still has a placeholder project: {PROJECT!r}. "
        "Edit configs/pipeline_config.json first."
    )

print("Will build and push:")
print(" ", IMAGE_URI)

Project root: /home/jupyter/mlops-meta-learning/fsl-projekt
Will build and push:
  us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train:candidate


## 2. Build + push (one run)

Ensures the Artifact Registry repo exists, authenticates Docker, builds, pushes.
The build step is the slow one (installs torch) - but Docker caches layers, so
if only `src/fsl/` changed since last time, it's fast; only a `pyproject.toml`
change triggers a full dependency reinstall.

In [2]:
# ensure the Docker-format repo exists (harmless if it already does)
!gcloud artifacts repositories describe {REPO} --location={REGION} --project={PROJECT} >/dev/null 2>&1 || gcloud artifacts repositories create {REPO} --repository-format=docker --location={REGION} --project={PROJECT}

# let Docker authenticate to Artifact Registry (safe to re-run)
!gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet

# build (from ROOT, so the Dockerfile + src/ + scripts/ are the build context)
!docker build -t {IMAGE_URI} .

# push
!docker push {IMAGE_URI}

print()
print("Done. Image pushed:")
print(" ", IMAGE_URI)


{
  "credHelpers": {
    "gcr.io": "gcloud",
    "us.gcr.io": "gcloud",
    "eu.gcr.io": "gcloud",
    "asia.gcr.io": "gcloud",
    "staging-k8s.gcr.io": "gcloud",
    "marketplace.gcr.io": "gcloud",
    "us-central1-docker.pkg.dev": "gcloud"
  }
}
Adding credentials for: us-central1-docker.pkg.dev
gcloud credential helpers already registered correctly.
[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 5.48kB                                     0.0s
 => resolve image config for docker-image://docker.io/docker/dockerfile:1  0.2s
[+] Building 0.3s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 5.48kB                                     0.0s
 =>

## 3. (optional) Confirm it's there / paste into config

Lists the images in the repo so you can see the tag landed. If your
`serving_container_image_uri` or the pipeline's training image should point at
this, here's the exact line to paste into `configs/pipeline_config.json`.

In [3]:
!gcloud artifacts docker images list {REGION}-docker.pkg.dev/{PROJECT}/{REPO} --include-tags 2>/dev/null | head -20

print()
print("URI to reference in config / pipeline:")
print(f'  "serving_container_image_uri": "{IMAGE_URI}"')

IMAGE                                                            DIGEST                                                                   TAGS        CREATE_TIME          UPDATE_TIME          SIZE
us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train  sha256:1f9c8c540be10a42c998d0ab57135e52b841c9f0dd5d41d29c0c4b4f1fbe2b69  candidate   2026-07-23T21:58:55  2026-07-23T21:58:55  3097499518
us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train  sha256:244e7b937d27c7c6494818840647a6cc7cf1484c2907d9e121c127f5da68a83d              2026-07-08T20:19:44  2026-07-08T22:13:10  3102318472
us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train  sha256:2f4db1764a3c44ee56605646c652edc24abd89f99a5ec0731382873ca4870bab              2026-07-23T15:05:51  2026-07-23T15:39:47  3097499634
us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train  sha256:59f9b3efd14dce60c4362d6f7d7707bd1eb399bd57d2514ebca6a049bafc272b              2026-07-23T15:39:47  2026-07-23T21:58:55  30

In [4]:
!docker run --rm --entrypoint ls us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train:candidate scripts/

__pycache__
baseline_frozen.py
bootstrap_fsl_data.py
build_image.ipynb
evaluate_pipeline_entry.py
explain_pipeline_entry.py
hpo_train_entry.py
publish_template.py
setup_env.sh
train.py
train_pipeline_entry.py
verify_frozen.py


In [5]:
!docker run --rm --entrypoint python \
  us-central1-docker.pkg.dev/dark-data-discovery/fsl-images/train:candidate \
  -c "from fsl.data.manifest import get_geometry; from fsl.data.registry import get_loaders; print('nowe moduly w obrazie: OK')"

nowe moduly w obrazie: OK


In [6]:
!gcloud logging read 'resource.labels.job_id="8305495615408701440"' \
  --project dark-data-discovery --limit 300 \
  --format 'value(textPayload)' > /tmp/job.log
grep -B 3 -A 40 "Traceback\|ComponentError\|Error" /tmp/job.log | head -80

SyntaxError: invalid syntax (168949329.py, line 2)

## Notes

- **Rebuild when code changes.** Because the pipeline uses this image (not
  inlined code), a change in `src/fsl/` only reaches the pipeline after you
  rerun cell 2. That's the trade for having the code in one place instead of
  duplicated across components.
- **`candidate` vs a version number.** `candidate` overwrites each build - it's
  "the current thing under test". If you want a permanent, non-overwriting tag
  (e.g. for a specific milestone), change `VERSION` to `v1`, `v2`, etc. The
  candidate/production flow doesn't need that, but it's there if you want it.
- **Speed.** First build is slow (torch). Later builds reuse cached layers
  unless `pyproject.toml` changed.